In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "workspace"
SCHEMA = "industrial_iot"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_equipment_telemetry"

ROW_COUNT = 1_000_000
MACHINE_COUNT = 200
BASE_EPOCH = 1735689600

print("Bronze table:", BRONZE_TABLE)
print("Rows to generate:", f"{ROW_COUNT:,}")

In [0]:
telemetry_base = (
    spark.range(ROW_COUNT)
    .withColumnRenamed("id", "event_id")
    .withColumn(
        "machine_id",
        F.concat(
            F.lit("M-"),
            F.lpad(
                ((F.col("event_id") % MACHINE_COUNT) + 1).cast("string"),
                3,
                "0",
            ),
        ),
    )
    .withColumn(
        "plant_id",
        F.concat(
            F.lit("P-"),
            F.lpad(
                ((F.col("event_id") % 5) + 1).cast("string"),
                2,
                "0",
            ),
        ),
    )
    .withColumn(
        "equipment_type",
        F.when((F.col("event_id") % 4) == 0, "Pump")
        .when((F.col("event_id") % 4) == 1, "Compressor")
        .when((F.col("event_id") % 4) == 2, "Motor")
        .otherwise("Turbine"),
    )
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.from_unixtime(
                F.lit(BASE_EPOCH) + (F.col("event_id") * 60)
            )
        ),
    )
)

print("Base telemetry created")
display(telemetry_base.limit(5))

In [0]:
telemetry = (
    telemetry_base
    .withColumn(
        "air_temperature_c",
        F.round(20 + F.rand(11) * 15, 2),
    )
    .withColumn(
        "process_temperature_c",
        F.round(F.col("air_temperature_c") + 5 + F.rand(12) * 10, 2),
    )
    .withColumn(
        "rotational_speed_rpm",
        F.round(1200 + F.rand(13) * 1800).cast("int"),
    )
    .withColumn(
        "torque_nm",
        F.round(5 + F.rand(14) * 75, 2),
    )
    .withColumn(
        "tool_wear_min",
        F.round(F.rand(15) * 250).cast("int"),
    )
    .withColumn(
        "vibration_mm_s",
        F.round(1 + F.rand(16) * 8, 3),
    )
    .withColumn(
        "pressure_bar",
        F.round(2 + F.rand(17) * 8, 2),
    )
    .withColumn(
        "failure_flag",
        (
            (F.col("vibration_mm_s") > 8.6)
            | (F.col("process_temperature_c") > 47.0)
            | (
                (F.col("tool_wear_min") > 240)
                & (F.col("torque_nm") > 72)
            )
        ).cast("int"),
    )
    .withColumn(
        "source_system",
        F.lit("synthetic_iot_gateway"),
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp(),
    )
)

print("Sensor readings created")

display(telemetry.limit(5))

telemetry.groupBy("failure_flag").count().orderBy("failure_flag").show()

In [0]:
(
    telemetry.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

bronze_df = spark.table(BRONZE_TABLE)
actual_rows = bronze_df.count()

print(f"Bronze table created: {BRONZE_TABLE}")
print(f"Rows written: {actual_rows:,}")

assert actual_rows == ROW_COUNT

spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}").select(
    "format",
    "numFiles",
    "sizeInBytes",
).show(truncate=False)

display(bronze_df.limit(10))